# AiPayGen Demo — x402 Paying Agent

This notebook demonstrates how AI agents pay for API calls using the x402 protocol.

**Prerequisites:** `pip install x402 eth-account requests`

**Cost:** ~$0.10 USDC total for all demos.

In [ ]:
import os, json, requests
from eth_account import Account
from x402 import x402ClientSync
from x402.mechanisms.evm.exact import register_exact_evm_client
from x402.mechanisms.evm.signers import EthAccountSigner
from x402.http.clients.requests import wrapRequestsWithPayment

# Set your private key (or use env var)
PRIVATE_KEY = os.getenv("AGENT_PRIVATE_KEY", "0xYOUR_KEY_HERE")
API = "https://api.aipaygen.com"

account = Account.from_key(PRIVATE_KEY)
signer = EthAccountSigner(account)
client = x402ClientSync()
register_exact_evm_client(client, signer)
session = wrapRequestsWithPayment(requests.Session(), client)

print(f"Wallet: {account.address}")
total_cost = 0.0

## Step 1: Single API Call — Research

In [ ]:
resp = session.post(f"{API}/research", json={"topic": "x402 protocol", "depth": "quick"}, timeout=60)
cost = float(resp.headers.get("X-Payment-Amount", "0"))
total_cost += cost
print(f"Status: {resp.status_code} | Cost: ${cost:.4f}")
research = resp.json()
print(json.dumps(research, indent=2)[:500])

## Step 2: Chain — Summarize + Translate

In [ ]:
# Summarize the research
resp = session.post(f"{API}/summarize", json={"text": json.dumps(research)[:2000], "style": "bullet_points"}, timeout=60)
cost = float(resp.headers.get("X-Payment-Amount", "0"))
total_cost += cost
print(f"Summarize — Status: {resp.status_code} | Cost: ${cost:.4f}")
summary = resp.json()
print(json.dumps(summary, indent=2)[:500])

# Translate
resp = session.post(f"{API}/translate", json={"text": json.dumps(summary)[:1000], "target_language": "French"}, timeout=60)
cost = float(resp.headers.get("X-Payment-Amount", "0"))
total_cost += cost
print(f"Translate — Status: {resp.status_code} | Cost: ${cost:.4f}")
print(resp.json())

## Step 3: Workflow — 3 Tools Chained (15% Discount)

In [ ]:
resp = session.post(f"{API}/workflow/run", json={
    "steps": [
        {"tool": "research", "input": {"topic": "AI micropayments"}},
        {"tool": "summarize", "input": {"style": "executive_summary"}},
        {"tool": "translate", "input": {"target_language": "Spanish"}},
    ]
}, timeout=120)
cost = float(resp.headers.get("X-Payment-Amount", "0"))
total_cost += cost
print(f"Workflow cost: ${cost:.4f} (15% discount)")
print(json.dumps(resp.json(), indent=2)[:500])

## Summary

In [ ]:
print(f"{'='*40}")
print(f"Total spent: ${total_cost:.4f} USDC")
print(f"{'='*40}")